# HAR LightGBM + Markov (目標 0.81+)
特徵工程同 0.79 baseline，加上 Markov 轉移後處理。

In [ ]:
import numpy as np
import pandas as pd
import os, glob
from scipy import stats
from scipy.fft import fft
from scipy.stats import entropy as sp_entropy
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

SEED      = 42
N_FOLDS   = 5
N_CLASSES = 6
CHANNELS  = ['mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z']

BASE_DIR   = '/Users/patrick/Desktop/528/nycu-data-mining-assignment-3'
BASE_TRAIN = os.path.join(BASE_DIR, 'train', 'train')
BASE_TEST  = os.path.join(BASE_DIR, 'test',  'test')
SAMPLE_SUB = os.path.join(BASE_DIR, 'sample_submission.csv')
LABEL_COL  = 'label'

## 1. 載入資料

In [ ]:
def load_split(base_dir, is_train=True):
    all_dfs   = []
    csv_files = sorted(glob.glob(os.path.join(base_dir, '*', '*.csv')))
    print(f'Found {len(csv_files)} window files')
    for fpath in csv_files:
        parts   = fpath.replace('\\', '/').split('/')
        user_id = parts[-2]
        win_id  = parts[-1][:-4]
        df = pd.read_csv(fpath, header=0)
        df.columns = df.columns.str.strip().str.lower()
        df['window_id'] = win_id
        df['user_id']   = user_id
        all_dfs.append(df)
    raw = pd.concat(all_dfs, ignore_index=True)
    if is_train:
        raw[LABEL_COL] = raw[LABEL_COL].astype(int)
    return raw

train_raw = load_split(BASE_TRAIN, is_train=True)
test_raw  = load_split(BASE_TEST,  is_train=False)
print(f'Train: {train_raw.shape}, Test: {test_raw.shape}')
print(train_raw.drop_duplicates('window_id')[LABEL_COL].value_counts().sort_index())

## 2. 特徵工程

In [ ]:
def summary_stats(arr, prefix):
    return {
        f'{prefix}_mean'    : np.mean(arr),
        f'{prefix}_std'     : np.std(arr),
        f'{prefix}_min'     : np.min(arr),
        f'{prefix}_max'     : np.max(arr),
        f'{prefix}_median'  : np.median(arr),
        f'{prefix}_q25'     : np.percentile(arr, 25),
        f'{prefix}_q75'     : np.percentile(arr, 75),
        f'{prefix}_kurtosis': stats.kurtosis(arr),
        f'{prefix}_skew'    : stats.skew(arr),
    }

def fft_features(arr, prefix, n_low=5):
    N      = len(arr)
    freqs  = np.abs(fft(arr))[:N//2]
    power  = freqs ** 2
    p_norm = power / (power.sum() + 1e-10)
    return {
        f'{prefix}_fft_low_power': np.sum(power[:n_low]),
        f'{prefix}_fft_peak_freq': float(np.argmax(power)),
        f'{prefix}_fft_entropy'  : sp_entropy(p_norm + 1e-10),
    }

def extract_window_features(df_win, user_position):
    feats = {}
    for ch in CHANNELS:
        feats.update(summary_stats(df_win[ch].values, ch))
    mx, my, mz = df_win['mean_x'].values, df_win['mean_y'].values, df_win['mean_z'].values
    sx, sy, sz = df_win['std_x'].values,  df_win['std_y'].values,  df_win['std_z'].values
    vm_mean = np.sqrt(mx**2 + my**2 + mz**2)
    vm_std  = np.sqrt(sx**2 + sy**2 + sz**2)
    feats.update(summary_stats(vm_mean, 'vm_mean'))
    feats.update(summary_stats(vm_std,  'vm_std'))
    feats.update(summary_stats(np.abs(vm_mean - 1.0), 'grav_delta'))
    n_seg, seg_len = 5, len(df_win) // 5
    for ch in CHANNELS:
        arr = df_win[ch].values
        for i in range(n_seg):
            seg = arr[i*seg_len:(i+1)*seg_len]
            feats[f'{ch}_seg{i}_mean']  = np.mean(seg)
            feats[f'{ch}_seg{i}_std']   = np.std(seg)
            feats[f'{ch}_seg{i}_range'] = np.max(seg) - np.min(seg)
    for ch in CHANNELS:
        d = np.diff(df_win[ch].values)
        feats[f'{ch}_diff_mean']     = np.mean(d)
        feats[f'{ch}_diff_std']      = np.std(d)
        feats[f'{ch}_diff_abs_mean'] = np.mean(np.abs(d))
    for ch in CHANNELS:
        feats.update(fft_features(df_win[ch].values, ch))

    # 跨 channel 相關係數 (3 對)
    for a, b in [('mean_x','mean_y'), ('mean_x','mean_z'), ('mean_y','mean_z')]:
        c = np.corrcoef(df_win[a].values, df_win[b].values)[0,1]
        feats[f'corr_{a}_{b}'] = float(c) if np.isfinite(c) else 0.0

    # 自相關係數 lag=1,5,10 (mean_x/y/z)
    for ch in ['mean_x', 'mean_y', 'mean_z']:
        s = df_win[ch].values - df_win[ch].mean()
        norm = np.dot(s, s) + 1e-10
        for lag in [1, 5, 10]:
            feats[f'{ch}_acf{lag}'] = float(np.dot(s[:-lag], s[lag:]) / norm)

    feats['user_position'] = user_position
    return feats

def build_features(raw_df):
    keep = CHANNELS + ['window_id', 'user_id']
    if LABEL_COL in raw_df.columns:
        keep.append(LABEL_COL)
    raw_df = raw_df[[c for c in keep if c in raw_df.columns]].copy()
    meta = raw_df.drop_duplicates('window_id')[['window_id', 'user_id']].copy()
    meta = meta.sort_values(['user_id', 'window_id'])
    meta['user_position'] = meta.groupby('user_id').cumcount()
    grp_size = meta.groupby('user_id')['window_id'].transform('count') - 1
    meta['user_position'] = meta['user_position'] / grp_size.clip(lower=1)
    pos_map = meta.set_index('window_id')['user_position'].to_dict()
    rows = []
    for wid, df_win in raw_df.groupby('window_id'):
        feat = extract_window_features(df_win, pos_map.get(wid, 0.5))
        feat['window_id'] = wid
        feat['user_id']   = df_win['user_id'].iloc[0]
        rows.append(feat)
    return pd.DataFrame(rows)

print('Building train features...')
train_feat = build_features(train_raw)
print('Building test features...')
test_feat  = build_features(test_raw)
label_map  = train_raw.drop_duplicates('window_id').set_index('window_id')[LABEL_COL]
train_feat['label'] = train_feat['window_id'].map(label_map)
print(f'Train: {train_feat.shape}, Test: {test_feat.shape}')

## 3. 準備 X, y + Class Weights

In [ ]:
EXCLUDE = {'window_id', 'user_id', 'label', 'index', 'file_id'}
FEATURE_COLS = [c for c in train_feat.columns if c not in EXCLUDE]
print(f'Feature count: {len(FEATURE_COLS)}')

X      = train_feat[FEATURE_COLS].values.astype(np.float32)
y      = train_feat['label'].values
groups = train_feat['user_id'].values
X_test = test_feat[FEATURE_COLS].values.astype(np.float32)

# Inverse-frequency class weights
_counts = np.bincount(y, minlength=N_CLASSES).astype(float)
_w = _counts.sum() / (N_CLASSES * _counts)
CLASS_WEIGHT = dict(enumerate(_w / _w.min()))
print('Class weights:', {k: f'{v:.2f}' for k, v in CLASS_WEIGHT.items()})
sample_weights = np.array([CLASS_WEIGHT[lbl] for lbl in y])

LGBM_PARAMS = dict(
    objective         = 'multiclass',
    num_class         = N_CLASSES,
    num_leaves        = 63,
    learning_rate     = 0.03,
    n_estimators      = 1000,   # 600 → 1000
    min_child_samples = 10,
    reg_alpha         = 0.05,
    reg_lambda        = 0.5,
    class_weight      = CLASS_WEIGHT,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

## 4. StratifiedGroupKFold + OOF

In [ ]:
sgkf      = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(X), N_CLASSES), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y, groups)):
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]
    sw_tr      = sample_weights[tr_idx]
    clf = LGBMClassifier(**LGBM_PARAMS)
    clf.fit(X_tr, y_tr, sample_weight=sw_tr)
    oof_proba[va_idx] = clf.predict_proba(X_va)
    fold_f1 = f1_score(y_va, np.argmax(oof_proba[va_idx], axis=1), average='macro')
    print(f'Fold {fold+1}/{N_FOLDS}  macro-F1: {fold_f1:.4f}')

base_f1 = f1_score(y, np.argmax(oof_proba, axis=1), average='macro')
print(f'\nOOF macro-F1 (before calibration): {base_f1:.4f}')

## 5. OOF Class-weight 校正

In [ ]:
WEIGHT_CANDIDATES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.5, 1.8, 2.0, 2.5]

current_weights = np.ones(N_CLASSES)
best_f1 = f1_score(y, np.argmax(oof_proba, axis=1), average='macro')
print(f'Baseline OOF F1 = {best_f1:.4f}')

for cls in range(N_CLASSES):
    best_w = current_weights[cls]
    for w in WEIGHT_CANDIDATES:
        trial      = current_weights.copy()
        trial[cls] = w
        f1 = f1_score(y, np.argmax(oof_proba * trial, axis=1), average='macro')
        if f1 > best_f1:
            best_f1 = f1
            best_w  = w
    current_weights[cls] = best_w
    print(f'  class {cls}: w={best_w:.2f}  running F1={best_f1:.4f}')

best_weights = current_weights.copy()
print(f'\nOOF macro-F1 after calibration: {best_f1:.4f}')
print(f'Calibrated weights: {best_weights}')

## 6. Full-train + **Fold Ensemble**

兩種預測方式：
- `full_clf`：用全部資料訓練一個模型（v2 方式）
- `fold_ensemble`：直接平均 5 個 fold model 的預測機率（ensemble，通常更穩）

In [ ]:
# 5-fold CV for OOF calibration only
fold_clfs = []
sgkf      = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_proba = np.zeros((len(X), N_CLASSES), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(sgkf.split(X, y, groups)):
    X_tr, y_tr = X[tr_idx], y[tr_idx]
    X_va, y_va = X[va_idx], y[va_idx]
    sw_tr      = sample_weights[tr_idx]
    clf = LGBMClassifier(**LGBM_PARAMS)
    clf.fit(X_tr, y_tr, sample_weight=sw_tr)
    fold_clfs.append(clf)
    oof_proba[va_idx] = clf.predict_proba(X_va)
    fold_f1 = f1_score(y_va, np.argmax(oof_proba[va_idx], axis=1), average='macro')
    print(f'Fold {fold+1}/{N_FOLDS}  macro-F1: {fold_f1:.4f}')

base_f1 = f1_score(y, np.argmax(oof_proba, axis=1), average='macro')
print(f'\nOOF macro-F1 (before calibration): {base_f1:.4f}')

# Multi-seed full model ensemble (5 seeds × full data)
SEEDS = [42, 123, 456, 789, 2024]
print(f'\nTraining {len(SEEDS)} full models (multi-seed ensemble)...')
full_clfs = []
for s in SEEDS:
    params = {**LGBM_PARAMS, 'random_state': s}
    m = LGBMClassifier(**params)
    m.fit(X, y, sample_weight=sample_weights)
    full_clfs.append(m)
    print(f'  seed={s} done')
print('All done.')

## 7. Markov 轉移後處理

按 user + window_id 排序後，對每個 file 的 proba 與上一個 file 的轉移機率混合。

真實轉移矩陣（從完整訓練資料計算）：

In [ ]:
TRANSITION = np.array([
    [0.987, 0.008, 0.001, 0.003, 0.000, 0.001],
    [0.008, 0.910, 0.026, 0.049, 0.002, 0.006],
    [0.014, 0.346, 0.466, 0.117, 0.011, 0.045],
    [0.015, 0.329, 0.075, 0.442, 0.035, 0.104],
    [0.007, 0.113, 0.035, 0.127, 0.718, 0.000],
    [0.006, 0.065, 0.021, 0.124, 0.002, 0.783],
])
ALPHA_PER_CLASS = np.array([0.40, 0.30, 0.05, 0.05, 0.20, 0.20])

def apply_markov_adaptive(proba_arr, user_ids, transition, cw, alpha_per_class):
    preds, prev = [], {}
    for proba, uid in zip(proba_arr, user_ids):
        if uid in prev:
            alpha   = alpha_per_class[prev[uid]]
            blended = (1 - alpha) * proba + alpha * transition[prev[uid]]
        else:
            blended = proba
        label = int(np.argmax(blended * cw))
        preds.append(label)
        prev[uid] = label
    return np.array(preds)

# 按 user + window_id 排序（保持時序）
test_feat_sorted = test_feat.sort_values(['user_id', 'window_id']).reset_index(drop=True)
X_test_sorted    = test_feat_sorted[FEATURE_COLS].values.astype(np.float32)
user_ids_sorted  = test_feat_sorted['user_id'].values

# Multi-seed ensemble：平均 5 個 full model 的機率
proba_multiseed = np.mean([m.predict_proba(X_test_sorted) for m in full_clfs], axis=0)
pred_multiseed  = apply_markov_adaptive(proba_multiseed, user_ids_sorted, TRANSITION, best_weights, ALPHA_PER_CLASS)

print('Multi-seed ensemble prediction distribution:')
print(pd.Series(pred_multiseed).value_counts().sort_index())

## 8. 輸出 submission CSV

In [ ]:
sample = pd.read_csv(SAMPLE_SUB)
pred_dict = dict(zip(test_feat_sorted['window_id'].astype(int).values, pred_multiseed))
sample['Label'] = sample['Id'].map(pred_dict).fillna(0).astype(int)

OUT = '/Users/patrick/Desktop/528/submission_multiseed.csv'
sample.to_csv(OUT, index=False)
print(f'Saved → {OUT}')
print(sample['Label'].value_counts().sort_index())
sample.head(10)

## 附錄：OOF per-class F1 & 特徵重要性

In [ ]:
import matplotlib.pyplot as plt
oof_pred = np.argmax(oof_proba * best_weights, axis=1)
print(classification_report(y, oof_pred))

fi = pd.Series(full_clf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
plt.figure(figsize=(10, 7))
fi.head(30).plot(kind='barh')
plt.gca().invert_yaxis()
plt.title('Top 30 Feature Importances')
plt.tight_layout()
plt.show()